In [1]:
import pandas as pd

df = pd.read_csv("../../02_Data/raw/gamefound_list.csv")

In [2]:
df

,Unnamed: 0,name,creator,currencySymbol,campaignGoal,fundsGathered,backersCount,campaignStart,campaignEnd,description,...,playTime,playTimeDescription,playTimeUnit,pledgeManagerAvailability,fundedInSeconds,imageUrl,pledgeManagerSoftCloseDeadline,projectTags,enableShippingOnlyMode,originalType
0,0,Bloodstone,Druid City Games,$,50000.0,93311.94,469,2026-06-02T13:00:00Z,2026-06-19T22:00:00Z,Bloodstone throws you into Vanira’s deadly are...,...,60.0,NaN,0,NaN,7770,https://imgcdn.gamefound.com/projectimage/proj...,NaN,"[{'projectTagID': 1, 'isSystem': False, 'isVis...",NaN,1
1,1,Aeolis,Meeple Pug,€,NaN,33413.90,305,2026-06-02T15:00:00Z,2026-06-11T19:00:00Z,Aeolis is a fully cooperative kingdom-building...,...,60.0,NaN,0,NaN,0,https://imgcdn.gamefound.com/projectimage/proj...,NaN,"[{'projectTagID': 18, 'isSystem': False, 'isVi...",NaN,2
2,2,A Life,Skellig Games,€,10000.0,22326.05,197,2026-06-02T15:00:00Z,2026-06-25T18:00:00Z,"In ""A Life"" 1-4 players play through a lifetim...",...,60.0,NaN,0,NaN,3756,https://imgcdn.gamefound.com/projectimage/proj...,NaN,"[{'projectTagID': 30, 'isSystem': False, 'isVi...",NaN,1
3,3,Growing Season Deluxe Edition,Undigital,€,8000.0,14881.70,352,2026-06-02T16:00:00Z,2026-06-22T22:00:00Z,Turn a quiet field into a thriving farm in thi...,...,30.0,NaN,0,NaN,4725,https://imgcdn.gamefound.com/projectimage/proj...,NaN,"[{'projectTagID': 2, 'isSystem': False, 'isVis...",NaN,1
4,4,Stonesaga Second Printing,Open Owl Studios,$,25000.0,154683.02,1788,2026-05-26T15:00:00Z,2026-06-20T00:00:00Z,Build a civilization across epochs in a cooper...,...,90.0,NaN,0,NaN,1764,https://imgcdn.gamefound.com/projectimage/proj...,NaN,"[{'projectTagID': 1, 'isSystem': False, 'isVis...",NaN,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2585,2585,Spring and Autumn: Story of China,Mr. B. Games,$,30000.0,38009.00,219,2021-12-01T15:00:00Z,2022-01-04T22:00:00Z,Welcome to Spring and Autumn: Story of China! ...,...,120.0,NaN,0,NaN,182152,https://imgcdn.gamefound.com/projectimage/proj...,NaN,"[{'projectTagID': 3, 'isSystem': False, 'isVis...",NaN,1
2586,2586,Crossroads Inn: The Board Game,Klabater,€,20000.0,13420.63,158,2021-10-28T14:00:00Z,2021-11-30T15:00:00Z,Folks! We keep playing games about dungeons an...,...,120.0,NaN,0,NaN,0,https://imgcdn.gamefound.com/projectimage/proj...,NaN,"[{'projectTagID': 1, 'isSystem': False, 'isVis...",NaN,1
2587,2587,PACHAMAMA,SitDown,€,25000.0,16048.00,318,2021-10-19T17:00:00Z,2021-11-09T18:00:00Z,The Earth Goddess Pachamama constantly guides ...,...,60.0,NaN,0,NaN,0,https://imgcdn.gamefound.com/projectimage/proj...,NaN,"[{'projectTagID': 3, 'isSystem': False, 'isVis...",NaN,1
2588,2588,Taverns & Dragons (Canceled),Lord Raccoon Games,€,20000.0,20954.00,470,2021-10-12T13:00:00Z,2021-11-03T22:00:00Z,Gather magic mushrooms and dragon eggs in the ...,...,60.0,NaN,0,NaN,433495,https://imgcdn.gamefound.com/projectimage/proj...,NaN,"[{'projectTagID': 3, 'isSystem': False, 'isVis...",NaN,1


In [ ]:
import requests
import re
import time
from tqdm.notebook import tqdm

project_urls = df['project_url']
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
}

rewards_list= []
fail_list=[]
for url in tqdm(project_urls):
    print(f"[{url}] 데이터 수집 중...")
    
    res = requests.get(url, headers=headers)
    if res.status_code != 200:
        print(f"페이지 로드 실패: {res.status_code}")
        continue

    match = re.search(r'"projectID"\s*:\s*(\d+)', res.text, re.IGNORECASE)
    
    if match:
        projectID_id = int(match.group(1))
        
        api_url = f"https://gamefound.com/api/projectContents/getRewards?projectID={projectID_id}"
        api_res = requests.get(api_url, headers=headers)
        if api_res.status_code == 200:
            data = api_res.json()
            if data['data']!=None:
                rewards_data = data.get('data').get('rewards')
            else:
                fail_list.append(url)            

            for item in rewards_data:
                item_dict = {
                    "raw_url" : url+"\\rewards",
                    'anchorRelativeUrl': item.get('anchorRelativeUrl'),
                    'backgroundUrl': item.get('backgroundUrl'),                # 상세페이지 대표 이미지
                    'deliveryDateRemarks': item.get('deliveryDateRemarks'),
                    'estimatedDeliveryAt': item.get('estimatedDeliveryAt'),    # 예상 배송일
                    'hasDetails': item.get('hasDetails'),
                    'isExposed': item.get('isExposed'),
                    'isMostPopular': item.get('isMostPopular'),                # 가장 인기 있는 상품 여부
                    'purchasedCopiesCount': item.get('purchasedCopiesCount'),  # 구매(후원)된 수량
                    'additionalInfoUrl': item.get('additionalInfoUrl'),
                    'hasInstallmentsAvailable': item.get('hasInstallmentsAvailable'), # 분할 납부 가능 여부
                    'installmentCost': item.get('installmentCost'),
                    'installmentMinPayment': item.get('installmentMinPayment'),
                    'abstract': item.get('abstract'),                          # 상품 요약 설명
                    'categoryID': item.get('categoryID'),
                    'effectivePrice': item.get('effectivePrice'),              # 실제 결제 가격 (할인 적용 등)
                    'hasLimitedStock': item.get('hasLimitedStock'),            # 한정 수량 여부
                    'hasRetailerAccess': item.get('hasRetailerAccess'),
                    'hasSpecialAccess': item.get('hasSpecialAccess'),
                    'imageUrl': item.get('imageUrl'),                          # 상품 썸네일 이미지
                    'isDigital': item.get('isDigital'),                        # 디지털 상품 여부
                    'isDiscounted': item.get('isDiscounted'),                  # 할인 여부
                    'isFeatured': item.get('isFeatured'),                      # 추천/강조 상품 여부
                    'name': item.get('name'),                                  # 상품명
                    'price': item.get('price'),                                # 원래 가격
                    'productCanBePurchased': item.get('productCanBePurchased'), # 구매 가능 여부
                    'productCanBePurchasedDescription': item.get('productCanBePurchasedDescription'),
                    'productID': item.get('productID'),                        # 상품 고유 ID
                    'projectID': item.get('projectID'),                        # 프로젝트 고유 ID
                    'remainingStockLimit': item.get('remainingStockLimit'),    # 남은 수량
                    'url': item.get('url'),                                    # 상품 상세 페이지 URL (상대 경로인 경우가 많음)
                    'productState': item.get('productState')
                }
                rewards_list.append(item_dict)


        else:
            print(f"  {api_url} 오류 {api_res.status_code}")
            
    else:
        print(f"{url} projectID 검색 실패")

    time.sleep(1.5)

df1 = pd.DataFrame(rewards_list)

In [ ]:
df1.to_csv("../../02_Data/raw/rewards.csv", index=True, encoding="utf-8-sig")

In [7]:
df1

,raw_url,anchorRelativeUrl,backgroundUrl,deliveryDateRemarks,estimatedDeliveryAt,hasDetails,isExposed,isMostPopular,purchasedCopiesCount,additionalInfoUrl,...,isFeatured,name,price,productCanBePurchased,productCanBePurchasedDescription,productID,projectID,remainingStockLimit,url,productState
0,https://gamefound.com:443/en/projects/oomm-gam...,stonesaga-all-in-107951,https://imgcdn.gamefound.com/productimage/proj...,None,2026-12-01T00:00:00Z,False,False,False,211,None,...,True,Stonesaga All-In,359.0,True,None,107951,9678,None,/en/projects/oomm-games/stonesaga-second-print...,2
1,https://gamefound.com:443/en/projects/oomm-gam...,errata-pack-108768,https://imgcdn.gamefound.com/productimage/proj...,None,2026-12-18T00:00:00Z,False,False,False,1274,None,...,False,Errata Pack,7.5,True,None,108768,9678,None,/en/projects/oomm-games/stonesaga-second-print...,2
2,https://gamefound.com:443/en/projects/oomm-gam...,stonesaga-107938,https://imgcdn.gamefound.com/productimage/proj...,None,2026-12-01T00:00:00Z,False,False,False,78,None,...,False,Stonesaga,119.0,True,None,107938,9678,None,/en/projects/oomm-games/stonesaga-second-print...,2
3,https://gamefound.com:443/en/projects/oomm-gam...,stonesaga-expansions-107941,https://imgcdn.gamefound.com/productimage/proj...,None,2026-12-01T00:00:00Z,False,False,False,228,None,...,False,Stonesaga & Expansions,229.0,True,None,107941,9678,None,/en/projects/oomm-games/stonesaga-second-print...,2
4,https://gamefound.com:443/en/projects/studio-m...,new-content-reward-105788,https://imgcdn.gamefound.com/productimage/proj...,\n,2026-04-28T00:00:00Z,False,False,False,2451,None,...,True,New Content Reward,50.0,True,None,105788,8330,None,/en/projects/studio-midhall/beast-ashfall#/pro...,2
5,https://gamefound.com:443/en/projects/studio-m...,reserve-my-spot-110491,https://imgcdn.gamefound.com/productimage/proj...,None,None,False,False,False,132,None,...,False,Reserve My Spot!,1.0,True,None,110491,8330,None,/en/projects/studio-midhall/beast-ashfall#/pro...,2
6,https://gamefound.com:443/en/projects/studio-m...,ashfall-107865,https://imgcdn.gamefound.com/productimage/proj...,None,None,False,False,False,221,None,...,False,Ashfall,35.0,True,None,107865,8330,None,/en/projects/studio-midhall/beast-ashfall#/pro...,2
7,https://gamefound.com:443/en/projects/studio-m...,beast-new-content-reward-107802,https://imgcdn.gamefound.com/productimage/proj...,None,None,False,False,False,52,None,...,False,Beast + New Content Reward,122.0,True,None,107802,8330,None,/en/projects/studio-midhall/beast-ashfall#/pro...,2
8,https://gamefound.com:443/en/projects/studio-m...,beast-all-expansions-reward-107804,https://imgcdn.gamefound.com/productimage/proj...,None,None,False,False,False,372,None,...,False,Beast + All Expansions Reward,240.0,True,None,107804,8330,None,/en/projects/studio-midhall/beast-ashfall#/pro...,2


In [11]:
api_url

'https://gamefound.com/api/projectContents/getRewards?projectID=2401'

In [12]:
url

'https://gamefound.com:443/en/projects/le-sesame/vatrakill'

In [16]:
data

{'data': None, 'success': True, 'message': None}

In [21]:
data['data']!=None

False